In [2]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise ValueError("Не найден OPENROUTER_API_KEY. Установи переменную окружения.")

llm = ChatOpenAI(
    model="qwen3.8-27b",                      
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0,                                # для стабильного JSON в части 2
    extra_body={"reasoning": {"enabled": False}}, # выключаем размышление 
)

# Проверяем, что соединение работает
response = llm.invoke("Скажи одно слово: работает?")
print("Тип ответа:", type(response))
print("Содержимое:", response.content)
print("usage_metadata:", response.usage_metadata)

Тип ответа: <class 'langchain_core.messages.ai.AIMessage'>
Содержимое: Да
usage_metadata: {'input_tokens': 20, 'output_tokens': 2, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


## Часть 1. Базовая цепочка LangChain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Параметризованный промпт:
#    - system: роль модели
#    - human: переменная query (запрос) и language (язык ответа)
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Ты — ассистент интернет-магазина. Твоя задача — определить категорию товара "
     "по запросу пользователя и кратко ответить. Отвечай на языке: {language}."),
    ("human", "{query}"),
])

# LCEL-цепочка: промпт → LLM → парсер строки
chain = prompt | llm | StrOutputParser()

Сравнение: прямой вызов LLM vs вызов цепочки


In [ ]:
direct_response = llm.invoke("Определи категорию товара: ноутбук до 70 тысяч рублей")
chain_response = chain.invoke({
    "query": "Определи категорию товара: ноутбук до 70 тысяч рублей",
    "language": "русский",
})

print("Прямой вызов LLM:")
print("  тип:", type(direct_response))
print("  содержимое:", direct_response.content[:100], "...")
print()
print("Вызов цепочки:")
print("  тип:", type(chain_response))
print("  содержимое:", chain_response[:100], "...")

Прямой вызов LLM:
  тип: <class 'langchain_core.messages.ai.AIMessage'>
  содержимое: Категория товара: **Ноутбуки** (или **Компьютеры и ноутбуки**). ...

Вызов цепочки:
  тип: <class 'langchain_core.messages.base.TextAccessor'>
  содержимое: Ноутбуки ...


In [9]:
direct_response

AIMessage(content='Категория товара: **Ноутбуки** (или **Компьютеры и ноутбуки**).', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 29, 'total_tokens': 52, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 8.665e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 8.665e-05, 'upstream_inference_prompt_cost': 1.305e-05, 'upstream_inference_completions_cost': 7.36e-05}}, 'model_provider': 'openai', 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': None, 'id': 'gen-1794333158-L0sWwiKBWSNa8wRspbUY', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1c697-2cd7-76a6-b94e-926afc92f238-0', tool_calls=[], inva

In [ ]:
print(type(chain_response).__mro__)   # посмотреть цепочку классов
print(isinstance(chain_response, str))  

(<class 'langchain_core.messages.base.TextAccessor'>, <class 'str'>, <class 'object'>)
True


In [5]:
queries = [
    {"query": "Найди ноутбуки дешевле 70 000 рублей", "language": "русский"},
    {"query": "Покажи самые дорогие смартфоны", "language": "русский"},
    {"query": "Хочу посмотреть планшеты", "language": "русский"},
]

batch_results = chain.batch(queries)

for q, r in zip(queries, batch_results):
    print(f"Запрос: {q['query']}")
    print(f"Ответ:  {r}\n")

Запрос: Найди ноутбуки дешевле 70 000 рублей
Ответ:  Ноутбуки

Запрос: Покажи самые дорогие смартфоны
Ответ:  Смартфоны

Запрос: Хочу посмотреть планшеты
Ответ:  Планшеты



## Часть 2. Structured Output и Pydantic

In [13]:
from typing import Literal, Optional
from pydantic import BaseModel, Field

class ProductQuery(BaseModel):
    category: Literal["laptop", "smartphone", "tablet", "other"] = Field(
        description="Категория товара. Если категория не входит в список поддерживаемых, используй other."
    )
    max_price: Optional[int] = Field(
        default=None,
        ge=0,
        description="Верхняя граница цены в рублях. Если ограничения нет — null."
    )
    sort_by: Literal["price_asc", "price_desc"] = Field(
        default="price_desc",
        description="Направление сортировки по цене: price_asc — по возрастанию, price_desc — по убыванию."
    )

In [14]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate

parser = PydanticOutputParser(pydantic_object=ProductQuery)
format_instructions = parser.get_format_instructions()

prompt_structured = ChatPromptTemplate.from_messages([
    ("system",
     "Ты — ассистент интернет-магазина. Преобразуй запрос пользователя в структурированные "
     "параметры поиска товаров.\n\n{format_instructions}"),
    ("human", "{query}"),
]).partial(format_instructions=format_instructions)

chain_structured = prompt_structured | llm | parser

In [15]:
result = chain_structured.invoke({"query": "Найди ноутбуки дешевле 70 000 рублей"})
print("Тип результата:", type(result))
print("Объект:", result)
print("JSON:", result.model_dump_json(indent=2))

Тип результата: <class '__main__.ProductQuery'>
Объект: category='laptop' max_price=70000 sort_by='price_asc'
JSON: {
  "category": "laptop",
  "max_price": 70000,
  "sort_by": "price_asc"
}


In [16]:
test_queries = [
    "Найди ноутбуки дешевле 70 000 рублей",
    "Покажи самые дорогие смартфоны",
    "Хочу посмотреть планшеты",
    "Покажи ноутбуки до 50 тысяч",
    "Найди холодильники",
]

for i, q in enumerate(test_queries, 1):
    print(f"Запрос {i}: {q}")
    try:
        result = chain_structured.invoke({"query": q})
        print(f"JSON: {result.model_dump_json(indent=2)}")
        print(f"Проверка полей:")
        print(f"  category  = {result.category!r} (допустимо: laptop/smartphone/tablet/other)")
        print(f"  max_price = {result.max_price!r} (None или неотрицательное целое)")
        print(f"  sort_by   = {result.sort_by!r} (price_asc / price_desc)")
    except Exception as e:
        print(f"ОШИБКА: {type(e).__name__}: {e}")

Запрос 1: Найди ноутбуки дешевле 70 000 рублей
JSON: {
  "category": "laptop",
  "max_price": 70000,
  "sort_by": "price_asc"
}
Проверка полей:
  category  = 'laptop' (допустимо: laptop/smartphone/tablet/other)
  max_price = 70000 (None или неотрицательное целое)
  sort_by   = 'price_asc' (price_asc / price_desc)
Запрос 2: Покажи самые дорогие смартфоны
JSON: {
  "category": "smartphone",
  "max_price": null,
  "sort_by": "price_desc"
}
Проверка полей:
  category  = 'smartphone' (допустимо: laptop/smartphone/tablet/other)
  max_price = None (None или неотрицательное целое)
  sort_by   = 'price_desc' (price_asc / price_desc)
Запрос 3: Хочу посмотреть планшеты
JSON: {
  "category": "tablet",
  "max_price": null,
  "sort_by": "price_desc"
}
Проверка полей:
  category  = 'tablet' (допустимо: laptop/smartphone/tablet/other)
  max_price = None (None или неотрицательное целое)
  sort_by   = 'price_desc' (price_asc / price_desc)
Запрос 4: Покажи ноутбуки до 50 тысяч
JSON: {
  "category": "lapt

In [17]:
ambiguous_queries = [
    "Что-нибудь недорогое для работы, но не телефон",
    "Покажи топовые варианты, но чтобы не слишком дорого",
]

for q in ambiguous_queries:
    print(f"Неоднозначный запрос: {q}")
    try:
        result = chain_structured.invoke({"query": q})
        print(f"JSON: {result.model_dump_json(indent=2)}")
    except Exception as e:
        print(f"ОШИБКА: {type(e).__name__}: {e}")

Неоднозначный запрос: Что-нибудь недорогое для работы, но не телефон
JSON: {
  "category": "laptop",
  "max_price": 50000,
  "sort_by": "price_asc"
}
Неоднозначный запрос: Покажи топовые варианты, но чтобы не слишком дорого
JSON: {
  "category": "other",
  "max_price": null,
  "sort_by": "price_desc"
}


 во втором вопросе - спрашивает "не слишком дорого" - а максимальной цены не поставила модель, зотя в первом случае ограничила 50000

модель не смогла извлечь категорию и числовое ограничение из формулировки без явных якорей: category="other", max_price=None. 

при отсутствии явных сущностей модель либо угадывает (как в первом запросе), либо откатывается к значениям по умолчанию. В реальной системе такие запросы следует распознавать и задавать пользователю уточняющий вопрос, а не выполнять поиск с other

In [18]:
try:
    result = chain_structured.invoke({"query": 'Что-нибудь до 30 тысяч'})
    print(f"JSON: {result.model_dump_json(indent=2)}")
except Exception as e:
    print(f"ОШИБКА: {type(e).__name__}: {e}")

JSON: {
  "category": "other",
  "max_price": 30000,
  "sort_by": "price_desc"
}


 число извлекла чётко, а категорию не угадывала, поставила other, потому что в тексте нет никаких указаний на тип товара

In [22]:
import json, os

os.makedirs("results", exist_ok=True)

test_queries = [
    "Найди ноутбуки дешевле 70 000 рублей",
    "Покажи самые дорогие смартфоны",
    "Хочу посмотреть планшеты",
    "Покажи ноутбуки до 50 тысяч",
    "Найди холодильники",
    "Что-нибудь недорогое для работы, но не телефон",
    "Покажи топовые варианты, но чтобы не слишком дорого",
    "Что-нибудь до 30 тысяч",
]

records = []
for q in test_queries:
    try:
        result = chain_structured.invoke({"query": q})
        records.append({
            "query": q,
            "json": json.loads(result.model_dump_json()),
            "valid": True,
            "error": None,
        })
        print(f"OK: {q} -> {result.model_dump_json()}")
    except Exception as e:
        records.append({
            "query": q,
            "json": None,
            "valid": False,
            "error": f"{type(e).__name__}: {e}",
        })
        print(f"FAIL: {q} -> {e}")

with open("results/part2_3_structured_tests.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"\nСохранено {len(records)} записей в results/part2_3_structured_tests.json")

OK: Найди ноутбуки дешевле 70 000 рублей -> {"category":"laptop","max_price":70000,"sort_by":"price_asc"}
OK: Покажи самые дорогие смартфоны -> {"category":"smartphone","max_price":null,"sort_by":"price_desc"}
OK: Хочу посмотреть планшеты -> {"category":"tablet","max_price":null,"sort_by":"price_desc"}
OK: Покажи ноутбуки до 50 тысяч -> {"category":"laptop","max_price":50000,"sort_by":"price_desc"}
OK: Найди холодильники -> {"category":"other","max_price":null,"sort_by":"price_desc"}
OK: Что-нибудь недорогое для работы, но не телефон -> {"category":"laptop","max_price":50000,"sort_by":"price_asc"}
OK: Покажи топовые варианты, но чтобы не слишком дорого -> {"category":"other","max_price":null,"sort_by":"price_desc"}
OK: Что-нибудь до 30 тысяч -> {"category":"other","max_price":30000,"sort_by":"price_desc"}

Сохранено 8 записей в results/part2_3_structured_tests.json


**Задание 2.4 — сравнение режима размышления**

Проверка: включается ли reasoning у текущей модели


In [19]:
llm_thinking = ChatOpenAI(
    model="qwen3.8-27b",   
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    extra_body={"reasoning": {"enabled": True}},
)

resp = llm_thinking.invoke("Сколько будет 17 * 23? Ответь одним числом.")
print("content:", repr(resp.content))
print("usage_metadata:", resp.usage_metadata)
print("response_metadata:", resp.response_metadata)

content: '391'
usage_metadata: {'input_tokens': 70, 'output_tokens': 58, 'total_tokens': 128, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 54}}
response_metadata: {'token_usage': {'completion_tokens': 58, 'prompt_tokens': 70, 'total_tokens': 128, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 54, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 0.000159, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000159, 'upstream_inference_prompt_cost': 1.4e-05, 'upstream_inference_completions_cost': 0.000145}}, 'model_provider': 'openai', 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': None, 'id': 'gen-1794335873-XcPcDyEdN26AOVmrzr4C', 'finish_reason'

In [20]:
import time

llm_off = ChatOpenAI(
    model="qwen3.8-27b", api_key=api_key, base_url="https://openrouter.ai/api/v1",
    temperature=0, extra_body={"reasoning": {"enabled": False}},
)
llm_on = ChatOpenAI(
    model="qwen3.8-27b", api_key=api_key, base_url="https://openrouter.ai/api/v1",
    temperature=0, extra_body={"reasoning": {"enabled": True}},
)

chain_off = prompt_structured | llm_off | parser
chain_on  = prompt_structured | llm_on  | parser

ambiguous = [
    "Что-нибудь недорогое для работы, но не телефон",
    "Покажи топовые варианты, но чтобы не слишком дорого",
]

for q in ambiguous:
    print(f"\n{'='*60}\nЗапрос: {q}")
    for name, chain in [("OFF", chain_off), ("ON", chain_on)]:
        t0 = time.time()
        try:
            result = chain.invoke({"query": q})
            elapsed = time.time() - t0
            print(f"\n[{name}] {elapsed:.2f} s")
            print(f"  JSON: {result.model_dump_json()}")
        except Exception as e:
            elapsed = time.time() - t0
            print(f"\n[{name}] {elapsed:.2f} s — ОШИБКА: {type(e).__name__}: {str(e)[:200]}")


Запрос: Что-нибудь недорогое для работы, но не телефон

[OFF] 2.74 s
  JSON: {"category":"laptop","max_price":50000,"sort_by":"price_asc"}

[ON] 0.76 s
  JSON: {"category":"laptop","max_price":null,"sort_by":"price_asc"}

Запрос: Покажи топовые варианты, но чтобы не слишком дорого

[OFF] 1.30 s
  JSON: {"category":"other","max_price":null,"sort_by":"price_desc"}

[ON] 4.05 s
  JSON: {"category":"other","max_price":null,"sort_by":"price_asc"}


посмотрим сырой content и токены

In [21]:
messages = prompt_structured.format_messages(query="Покажи топовые варианты, но чтобы не слишком дорого")
for name, llm_x in [("OFF", llm_off), ("ON", llm_on)]:
    t0 = time.time()
    resp = llm_x.invoke(messages)
    elapsed = time.time() - t0
    print(f"\n[{name}] {elapsed:.2f} s")
    print(f"  content: {resp.content!r}")
    print(f"  usage: {resp.usage_metadata}")


[OFF] 2.29 s
  content: '```json\n{\n  "category": "other",\n  "max_price": null,\n  "sort_by": "price_desc"\n}\n```'
  usage: {'input_tokens': 364, 'output_tokens': 35, 'total_tokens': 399, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

[ON] 4.87 s
  content: '\n\n{\n  "category": "other",\n  "max_price": null,\n  "sort_by": "price_asc"\n}'
  usage: {'input_tokens': 400, 'output_tokens': 394, 'total_tokens': 794, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 363}}


In [24]:
import json, os, time

os.makedirs("results", exist_ok=True)

ambiguous = [
    "Что-нибудь недорогое для работы, но не телефон",
    "Покажи топовые варианты, но чтобы не слишком дорого",
]

comparison = []
for q in ambiguous:
    messages = prompt_structured.format_messages(query=q)
    entry = {"query": q, "off": {}, "on": {}}
    for name, llm_x, chain_x in [("off", llm_off, chain_off), ("on", llm_on, chain_on)]:
        # 1) сырой вызов LLM — нужен для content и usage_metadata
        t0 = time.time()
        raw = llm_x.invoke(messages)
        raw_time = time.time() - t0

        # 2) вызов через цепочку — проверяем, что парсер справляется
        t0 = time.time()
        try:
            parsed = chain_x.invoke({"query": q})
            chain_time = time.time() - t0
            parsed_json = json.loads(parsed.model_dump_json())
            chain_error = None
        except Exception as e:
            chain_time = time.time() - t0
            parsed_json = None
            chain_error = f"{type(e).__name__}: {e}"

        entry[name] = {
            "raw_content": raw.content,
            "raw_time_sec": round(raw_time, 2),
            "chain_time_sec": round(chain_time, 2),
            "usage_metadata": raw.usage_metadata,
            "parsed_json": parsed_json,
            "chain_error": chain_error,
        }
        print(f"[{name}] {q[:40]}... -> {parsed_json}, {raw_time:.2f}s, "
              f"reasoning={raw.usage_metadata['output_token_details']['reasoning']}")
    comparison.append(entry)

with open("results/part2_4_reasoning_comparison.json", "w", encoding="utf-8") as f:
    json.dump(comparison, f, ensure_ascii=False, indent=2)

print("\nСохранено в results/part2_4_reasoning_comparison.json")

[off] Что-нибудь недорогое для работы, но не т... -> {'category': 'laptop', 'max_price': 50000, 'sort_by': 'price_asc'}, 1.66s, reasoning=0
[on] Что-нибудь недорогое для работы, но не т... -> {'category': 'laptop', 'max_price': None, 'sort_by': 'price_asc'}, 3.33s, reasoning=411
[off] Покажи топовые варианты, но чтобы не сли... -> {'category': 'other', 'max_price': None, 'sort_by': 'price_desc'}, 0.74s, reasoning=0
[on] Покажи топовые варианты, но чтобы не сли... -> {'category': 'other', 'max_price': None, 'sort_by': 'price_asc'}, 9.24s, reasoning=596

Сохранено в results/part2_4_reasoning_comparison.json


## Часть 3. Вызов классического программного обеспечения

In [27]:
import importlib, search
importlib.reload(search)
from search import ProductQuery, search_products

tests = [
    ProductQuery(category="laptop", max_price=70000, sort_by="price_asc"),
    ProductQuery(category="smartphone", sort_by="price_desc"),
    ProductQuery(category="tablet", sort_by="price_asc"),
    ProductQuery(category="laptop", max_price=30000, sort_by="price_asc"),
    ProductQuery(category="other", sort_by="price_desc"),
]

for q in tests:
    res = search_products(q)
    print(f"\n{q}")
    print(f"  найдено: {len(res)}")
    for r in res:
        print(f"    {r['name']} — {r['price']} ₽")


category='laptop' max_price=70000 sort_by='price_asc'
  найдено: 3
    Lenovo IdeaPad 3 — 45000 ₽
    ASUS VivoBook 15 — 58000 ₽
    Acer Aspire 5 — 62000 ₽

category='smartphone' max_price=None sort_by='price_desc'
  найдено: 6
    Apple iPhone 15 Pro — 130000 ₽
    Samsung Galaxy S23 — 95000 ₽
    Apple iPhone 14 — 78000 ₽
    Google Pixel 7a — 45000 ₽
    Samsung Galaxy A54 — 32000 ₽
    Xiaomi Redmi Note 12 — 18000 ₽

category='tablet' max_price=None sort_by='price_asc'
  найдено: 5
    Lenovo Tab M10 — 22000 ₽
    Samsung Galaxy Tab A8 — 30000 ₽
    Xiaomi Pad 6 — 42000 ₽
    Apple iPad 10 — 55000 ₽
    Samsung Galaxy Tab S9 — 85000 ₽

category='laptop' max_price=30000 sort_by='price_asc'
  найдено: 0

category='other' max_price=None sort_by='price_desc'
  найдено: 0


При category="other" функция search_products возвращает пустой список и не выполняет SQL-запрос по всему каталогу.

In [28]:
from langchain_core.runnables import RunnableLambda
from search import search_products

def run_search(query: ProductQuery) -> dict:
    """Принимает валидированный ProductQuery, возвращает результат поиска."""
    if query.category == "other":
        return {
            "query": query.model_dump(),
            "products": [],
            "message": "Категория не поддерживается. Уточните запрос.",
        }
    products = search_products(query)
    return {
        "query": query.model_dump(),
        "products": products,
        "message": None if products else "Ничего не найдено.",
    }

chain_full = chain_structured | RunnableLambda(run_search)

In [29]:
tests = [
    "Покажи самые дешевые ноутбуки до 80 тысяч рублей",
    "Покажи самые дорогие смартфоны",
    "Хочу посмотреть планшеты",
    "Найди холодильники",
    "Найди ноутбуки дешевле 30 тысяч",
]

for q in tests:
    print(f"\n{'='*60}\nЗапрос: {q}")
    out = chain_full.invoke({"query": q})
    print(f"Параметры: {out['query']}")
    if out["message"]:
        print(f"Сообщение: {out['message']}")
    for p in out["products"]:
        print(f"  {p['name']} — {p['price']} ₽")


Запрос: Покажи самые дешевые ноутбуки до 80 тысяч рублей
Параметры: {'category': 'laptop', 'max_price': 80000, 'sort_by': 'price_asc'}
  Lenovo IdeaPad 3 — 45000 ₽
  ASUS VivoBook 15 — 58000 ₽
  Acer Aspire 5 — 62000 ₽
  HP Pavilion 14 — 75000 ₽

Запрос: Покажи самые дорогие смартфоны
Параметры: {'category': 'smartphone', 'max_price': None, 'sort_by': 'price_desc'}
  Apple iPhone 15 Pro — 130000 ₽
  Samsung Galaxy S23 — 95000 ₽
  Apple iPhone 14 — 78000 ₽
  Google Pixel 7a — 45000 ₽
  Samsung Galaxy A54 — 32000 ₽
  Xiaomi Redmi Note 12 — 18000 ₽

Запрос: Хочу посмотреть планшеты
Параметры: {'category': 'tablet', 'max_price': None, 'sort_by': 'price_desc'}
  Samsung Galaxy Tab S9 — 85000 ₽
  Apple iPad 10 — 55000 ₽
  Xiaomi Pad 6 — 42000 ₽
  Samsung Galaxy Tab A8 — 30000 ₽
  Lenovo Tab M10 — 22000 ₽

Запрос: Найди холодильники
Параметры: {'category': 'other', 'max_price': None, 'sort_by': 'price_desc'}
Сообщение: Категория не поддерживается. Уточните запрос.

Запрос: Найди ноутбуки деш

In [30]:
import json, os

os.makedirs("results", exist_ok=True)

tests = [
    "Покажи самые дешевые ноутбуки до 80 тысяч рублей",
    "Покажи самые дорогие смартфоны",
    "Хочу посмотреть планшеты",
    "Найди холодильники",
    "Найди ноутбуки дешевле 30 тысяч",
]

records_33 = []
for q in tests:
    out = chain_full.invoke({"query": q})
    records_33.append({
        "query": q,
        "params": out["query"],
        "products": out["products"],
        "message": out["message"],
        "correct": True,  
    })

with open("results/part3_3_search_tests.json", "w", encoding="utf-8") as f:
    json.dump(records_33, f, ensure_ascii=False, indent=2)

print(f"Сохранено {len(records_33)} записей в results/part3_3_search_tests.json")

Сохранено 5 записей в results/part3_3_search_tests.json


In [31]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

plain_chain = ChatPromptTemplate.from_messages([
    ("system", "Ты — ассистент интернет-магазина. Отвечай кратко."),
    ("human", "{query}"),
]) | llm | StrOutputParser()

compare_queries = [
    "Какие ноутбуки дешевле 70 000 рублей есть в каталоге?",
    "Какие самые дорогие смартфоны есть в каталоге?",
    "Какие планшеты есть в каталоге?",
]

for q in compare_queries:
    print(f"\n{'='*60}\nВопрос: {q}")
    print("--- Без базы (только LLM): ---")
    print(plain_chain.invoke({"query": q}))
    print("--- С базой (пайплайн): ---")
    out = chain_full.invoke({"query": q})
    for p in out["products"]:
        print(f"  {p['name']} — {p['price']} ₽")


Вопрос: Какие ноутбуки дешевле 70 000 рублей есть в каталоге?
--- Без базы (только LLM): ---
Я не имею доступа к вашему текущему каталогу товаров. Пожалуйста, уточните, какой именно магазин вы имеете в виду, или предоставьте список доступных моделей.
--- С базой (пайплайн): ---
  Lenovo IdeaPad 3 — 45000 ₽
  ASUS VivoBook 15 — 58000 ₽
  Acer Aspire 5 — 62000 ₽

Вопрос: Какие самые дорогие смартфоны есть в каталоге?
--- Без базы (только LLM): ---
У меня нет доступа к вашему текущему каталогу товаров. Пожалуйста, уточните, какой именно магазин вы имеете в виду, или предоставьте список товаров.
--- С базой (пайплайн): ---
  Apple iPhone 15 Pro — 130000 ₽
  Samsung Galaxy S23 — 95000 ₽
  Apple iPhone 14 — 78000 ₽
  Google Pixel 7a — 45000 ₽
  Samsung Galaxy A54 — 32000 ₽
  Xiaomi Redmi Note 12 — 18000 ₽

Вопрос: Какие планшеты есть в каталоге?
--- Без базы (только LLM): ---
У меня нет доступа к вашему каталогу товаров. Пожалуйста, уточните, какой именно магазин вы имеете в виду, или предо

без базы 0 из 3 ответов корректны (все отказы), с базой 3 из 3 точных. Сравнение наглядное.

In [32]:
no_db_records = []
for q in compare_queries:
    no_db_answer = plain_chain.invoke({"query": q})
    with_db = chain_full.invoke({"query": q})
    no_db_records.append({
        "query": q,
        "answer_without_db": no_db_answer,
        "answer_with_db": with_db["products"],
        "params": with_db["query"],
    })

with open("results/part3_3_no_db_comparison.json", "w", encoding="utf-8") as f:
    json.dump(no_db_records, f, ensure_ascii=False, indent=2)

print(f"Сохранено {len(no_db_records)} сравнений в results/part3_3_no_db_comparison.json")

Сохранено 3 сравнений в results/part3_3_no_db_comparison.json


In [33]:
from langchain_core.tools import tool
from pydantic import Field

@tool(args_schema=ProductQuery)
def search_products_tool(query: ProductQuery) -> str:
    """Ищет товары в каталоге по категории, максимальной цене и сортировке."""
    if query.category == "other":
        return "Категория не поддерживается."
    products = search_products(query)
    if not products:
        return "Ничего не найдено."
    return "\n".join(f"{p['name']} — {p['price']} ₽" for p in products)

llm_with_tools = llm.bind_tools([search_products_tool])

In [34]:
tool_test_queries = [
    "Найди ноутбуки дешевле 70 000 рублей",
    "Покажи самые дорогие смартфоны",
    "Хочу посмотреть планшеты",
    "Покажи ноутбуки до 50 тысяч",
    "Найди холодильники",
]

for q in tool_test_queries:
    print(f"\n{'='*60}\nЗапрос: {q}")
    response = llm_with_tools.invoke(q)
    print(f"Текст ответа модели: {response.content!r}")
    print(f"tool_calls: {response.tool_calls}")


Запрос: Найди ноутбуки дешевле 70 000 рублей
Текст ответа модели: ''
tool_calls: [{'name': 'search_products_tool', 'args': {'category': 'laptop', 'max_price': 70000}, 'id': 'call_a61d7b358d2c4dcd96deec4b', 'type': 'tool_call'}]

Запрос: Покажи самые дорогие смартфоны
Текст ответа модели: ''
tool_calls: [{'name': 'search_products_tool', 'args': {'category': 'smartphone', 'sort_by': 'price_desc'}, 'id': 'call_7d4a6eabe1a242c78322f4e4664b741c', 'type': 'tool_call'}]

Запрос: Хочу посмотреть планшеты
Текст ответа модели: 'Вот список планшетов из каталога:\n\n'
tool_calls: [{'name': 'search_products_tool', 'args': {'category': 'tablet'}, 'id': 'chatcmpl-tool-8a1a6b1001e62fa7', 'type': 'tool_call'}]

Запрос: Покажи ноутбуки до 50 тысяч
Текст ответа модели: ''
tool_calls: [{'name': 'search_products_tool', 'args': {'category': 'laptop', 'max_price': 50000, 'sort_by': 'price_asc'}, 'id': 'call_28a5b3c377f94e7f9e5f6020', 'type': 'tool_call'}]

Запрос: Найди холодильники
Текст ответа модели: 'К 

In [35]:
from pydantic import ValidationError

def execute_tool_call(tool_call) -> dict:
    """Проверяет args по ProductQuery и выполняет поиск."""
    name = tool_call["name"]
    if name != "search_products_tool":
        return {"error": f"Неизвестный инструмент: {name}"}

    try:
        query = ProductQuery(**tool_call["args"])   # ← валидация!
    except ValidationError as e:
        return {"error": f"Аргументы не прошли валидацию: {e}"}

    if query.category == "other":
        return {"query": query.model_dump(), "products": [], "message": "Категория не поддерживается."}

    products = search_products(query)
    return {
        "query": query.model_dump(),
        "products": products,
        "message": None if products else "Ничего не найдено.",
    }

for q in tool_test_queries:
    print(f"\n{'='*60}\nЗапрос: {q}")
    response = llm_with_tools.invoke(q)

    if not response.tool_calls:
        print("Модель НЕ вызвала инструмент. Текст:", response.content)
        continue

    for tc in response.tool_calls:
        print(f"Вызов: {tc['name']}, args={tc['args']}")
        result = execute_tool_call(tc)
        print(f"Результат: {result}")


Запрос: Найди ноутбуки дешевле 70 000 рублей
Вызов: search_products_tool, args={'category': 'laptop', 'max_price': 70000, 'sort_by': 'price_asc'}
Результат: {'query': {'category': 'laptop', 'max_price': 70000, 'sort_by': 'price_asc'}, 'products': [{'id': 1, 'name': 'Lenovo IdeaPad 3', 'category': 'laptop', 'price': 45000}, {'id': 2, 'name': 'ASUS VivoBook 15', 'category': 'laptop', 'price': 58000}, {'id': 3, 'name': 'Acer Aspire 5', 'category': 'laptop', 'price': 62000}], 'message': None}

Запрос: Покажи самые дорогие смартфоны
Вызов: search_products_tool, args={'category': 'smartphone', 'sort_by': 'price_desc'}
Результат: {'query': {'category': 'smartphone', 'max_price': None, 'sort_by': 'price_desc'}, 'products': [{'id': 12, 'name': 'Apple iPhone 15 Pro', 'category': 'smartphone', 'price': 130000}, {'id': 11, 'name': 'Samsung Galaxy S23', 'category': 'smartphone', 'price': 95000}, {'id': 10, 'name': 'Apple iPhone 14', 'category': 'smartphone', 'price': 78000}, {'id': 9, 'name': 'Goo

In [36]:
response = llm_with_tools.invoke("Привет! Как дела?")
print("content:", response.content)
print("tool_calls:", response.tool_calls)

content: Привет! У меня всё отлично, спасибо! А у вас как дела?

Чем могу помочь? Если хотите, могу поискать для вас товары в каталоге — например, ноутбуки, смартфоны или планшеты.
tool_calls: []


In [37]:
import json

records_34 = []
for q in tool_test_queries:
    response = llm_with_tools.invoke(q)
    entry = {
        "query": q,
        "content": response.content,
        "tool_calls": response.tool_calls,
        "executed": [],
    }
    for tc in response.tool_calls:
        entry["executed"].append(execute_tool_call(tc))
    records_34.append(entry)

# отдельно — случай без вызова инструмента
resp_no_tool = llm_with_tools.invoke("Привет! Как дела?")
records_34.append({
    "query": "Привет! Как дела?",
    "content": resp_no_tool.content,
    "tool_calls": resp_no_tool.tool_calls,
    "executed": [],
    "note": "модель не вызвала инструмент",
})

with open("results/part3_4_tool_calls.json", "w", encoding="utf-8") as f:
    json.dump(records_34, f, ensure_ascii=False, indent=2)

print(f"Сохранено {len(records_34)} записей в results/part3_4_tool_calls.json")

Сохранено 6 записей в results/part3_4_tool_calls.json


## Часть 4. Retry и журнал выполнения

три невалидных ответа и прогон их через парсер, чтобы увидеть, какие исключения возникают и на каком этапе



In [38]:
from pydantic import ValidationError
import json

bad_outputs = {
    "1. Синтаксически некорректный JSON": '{"category": "laptop", "max_price": 70000,}',  # лишняя запятая
    "2. Значение не из перечисления": '{"category": "fridge", "max_price": 50000, "sort_by": "price_asc"}',
    "3. Неверный тип / нарушение ограничения": '{"category": "laptop", "max_price": -100, "sort_by": "price_asc"}',
}

for name, raw in bad_outputs.items():
    print(f"\n{'='*60}\n{name}")
    print(f"Сырой ответ: {raw!r}")
    try:
        parsed = parser.parse(raw)
        print(f"УСПЕХ (неожиданно): {parsed}")
    except Exception as e:
        print(f"Исключение: {type(e).__name__}")
        print(f"Сообщение: {str(e)[:300]}")
        cause = getattr(e, "__cause__", None)
        if cause:
            print(f"Причина (__cause__): {type(cause).__name__}: {str(cause)[:300]}")


1. Синтаксически некорректный JSON
Сырой ответ: '{"category": "laptop", "max_price": 70000,}'
Исключение: OutputParserException
Сообщение: Invalid json output: {"category": "laptop", "max_price": 70000,}
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 
Причина (__cause__): JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 43 (char 42)

2. Значение не из перечисления
Сырой ответ: '{"category": "fridge", "max_price": 50000, "sort_by": "price_asc"}'
Исключение: OutputParserException
Сообщение: Failed to parse ProductQuery from completion {"category": "fridge", "max_price": 50000, "sort_by": "price_asc"}. Got: 1 validation error for ProductQuery
category
  Input should be 'laptop', 'smartphone', 'tablet' or 'other' [type=literal_error, input_value='fridge', input_type=str]
    For further 
Причина (__cause__): ValidationError: 1 validation error for ProductQuery
category
  Input should be 'laptop', '

In [39]:
import json

errors_41 = []
for name, raw in bad_outputs.items():
    entry = {"case": name, "raw": raw, "exception": None, "cause": None, "message": None}
    try:
        parser.parse(raw)
        entry["exception"] = "NO ERROR (unexpected)"
    except Exception as e:
        entry["exception"] = type(e).__name__
        entry["message"] = str(e)[:500]
        cause = getattr(e, "__cause__", None)
        if cause:
            entry["cause"] = f"{type(cause).__name__}: {str(cause)[:500]}"
    errors_41.append(entry)

with open("results/part4_1_parse_errors.json", "w", encoding="utf-8") as f:
    json.dump(errors_41, f, ensure_ascii=False, indent=2)

print(f"Сохранено {len(errors_41)} ошибок в results/part4_1_parse_errors.json")

Сохранено 3 ошибок в results/part4_1_parse_errors.json


**with_retry**

In [ ]:
from langchain_core.runnables import RunnableLambda

# # Цепочка "промпт → LLM → парсер" с retry
# chain_structured_retry = (prompt_structured | llm | parser).with_retry(
#     stop_after_attempt=3,
#     retry_if_exception_type=(Exception,),   # сузим 
# )

from langchain_core.exceptions import OutputParserException

chain_structured_retry = (prompt_structured | llm | parser).with_retry(
    stop_after_attempt=3,
    retry_if_exception_type=(OutputParserException,),
)

# Полная цепочка: retry-часть + поиск (поиск НЕ повторяется)
chain_full_retry = chain_structured_retry | RunnableLambda(run_search)

In [42]:
out = chain_full_retry.invoke({"query": "Найди ноутбуки дешевле 70 000 рублей"})
print(out)

{'query': {'category': 'laptop', 'max_price': 70000, 'sort_by': 'price_asc'}, 'products': [{'id': 1, 'name': 'Lenovo IdeaPad 3', 'category': 'laptop', 'price': 45000}, {'id': 2, 'name': 'ASUS VivoBook 15', 'category': 'laptop', 'price': 58000}, {'id': 3, 'name': 'Acer Aspire 5', 'category': 'laptop', 'price': 62000}], 'message': None}


проверим действительно ли with_retry навешан только на генерацию+парсинг, а не на весь пайплайн

In [43]:
search_call_count = {"n": 0}

def run_search_counted(query: ProductQuery) -> dict:
    search_call_count["n"] += 1
    return run_search(query)

chain_full_retry_check = chain_structured_retry | RunnableLambda(run_search_counted)

# нормальный запрос — поиск должен выполниться ровно 1 раз
search_call_count["n"] = 0
out = chain_full_retry_check.invoke({"query": "Найди ноутбуки дешевле 70 000 рублей"})
print("Поиск вызван раз:", search_call_count["n"])   # ожидаем 1
print("Найдено:", len(out["products"]))

Поиск вызван раз: 1
Найдено: 3


In [44]:
def safe_invoke(query: str) -> dict:
    try:
        return chain_full_retry.invoke({"query": query})
    except OutputParserException as e:
        return {
            "query": None,
            "products": [],
            "message": f"Не удалось разобрать ответ модели после 3 попыток: {str(e)[:200]}",
        }

# проверка на нормальном запросе
print(safe_invoke("Найди ноутбуки дешевле 70 000 рублей"))

{'query': {'category': 'laptop', 'max_price': 70000, 'sort_by': 'price_asc'}, 'products': [{'id': 1, 'name': 'Lenovo IdeaPad 3', 'category': 'laptop', 'price': 45000}, {'id': 2, 'name': 'ASUS VivoBook 15', 'category': 'laptop', 'price': 58000}, {'id': 3, 'name': 'Acer Aspire 5', 'category': 'laptop', 'price': 62000}], 'message': None}


**журнал выполнения**

In [45]:
import json, time
from datetime import datetime

pipeline_log = []

def log(step: str, **details):
    entry = {"time": datetime.now().isoformat(timespec="seconds"), "step": step, **details}
    pipeline_log.append(entry)
    print(f"[LOG] {step}: {details}")

In [46]:
from langchain_core.runnables import RunnableLambda

attempt_counter = {"n": 0}

def llm_step(inputs):
    """Вызов LLM с логированием номера попытки."""
    attempt_counter["n"] += 1
    log("llm_call_start", attempt=attempt_counter["n"])
    t0 = time.time()
    resp = llm.invoke(inputs)
    log("llm_call_end", attempt=attempt_counter["n"], elapsed=round(time.time()-t0, 2),
        input_tokens=resp.usage_metadata["input_tokens"],
        output_tokens=resp.usage_metadata["output_tokens"])
    return resp

def parse_step(ai_message):
    """Парсинг ответа LLM с логированием результата или ошибки."""
    try:
        parsed = parser.invoke(ai_message)
        log("parse_ok", category=parsed.category, max_price=parsed.max_price, sort_by=parsed.sort_by)
        return parsed
    except Exception as e:
        log("parse_error", error_type=type(e).__name__, error=str(e)[:200])
        raise

chain_logged = prompt_structured | RunnableLambda(llm_step) | RunnableLambda(parse_step)

версия цепочки, которая логирует каждый этап

with_retry не даёт напрямую узнать номер попытки, поэтому обернем LLM в RunnableLambda, который считает попытки и пишет в лог, а парсер в функцию, которая логирует результат разбора или ошибку.

In [47]:
from langchain_core.runnables import RunnableLambda

attempt_counter = {"n": 0}

def llm_step(inputs):
    """Вызов LLM с логированием номера попытки."""
    attempt_counter["n"] += 1
    log("llm_call_start", attempt=attempt_counter["n"])
    t0 = time.time()
    resp = llm.invoke(inputs)
    log("llm_call_end", attempt=attempt_counter["n"], elapsed=round(time.time()-t0, 2),
        input_tokens=resp.usage_metadata["input_tokens"],
        output_tokens=resp.usage_metadata["output_tokens"])
    return resp

def parse_step(ai_message):
    """Парсинг ответа LLM с логированием результата или ошибки."""
    try:
        parsed = parser.invoke(ai_message)
        log("parse_ok", category=parsed.category, max_price=parsed.max_price, sort_by=parsed.sort_by)
        return parsed
    except Exception as e:
        log("parse_error", error_type=type(e).__name__, error=str(e)[:200])
        raise

chain_logged = prompt_structured | RunnableLambda(llm_step) | RunnableLambda(parse_step)

цепочка с retry и логированием, плюс шаг поиска с логированием

In [48]:
def run_search_logged(query: ProductQuery) -> dict:
    log("params_validated", params=query.model_dump())
    if query.category == "other":
        log("search_skipped", reason="unsupported_category")
        return {"query": query.model_dump(), "products": [], "message": "Категория не поддерживается."}
    products = search_products(query)
    log("search_done", found=len(products))
    return {"query": query.model_dump(), "products": products,
            "message": None if products else "Ничего не найдено."}

chain_logged_retry = chain_logged.with_retry(
    stop_after_attempt=3,
    retry_if_exception_type=(OutputParserException,),
)
chain_full_logged = chain_logged_retry | RunnableLambda(run_search_logged)

In [49]:
pipeline_log.clear()
attempt_counter["n"] = 0
result = chain_full_logged.invoke({"query": "Найди ноутбуки дешевле 70 000 рублей"})
print("\nРезультат:", result["query"], "найдено:", len(result["products"]))

print("\n--- Журнал ---")
for e in pipeline_log:
    print(json.dumps(e, ensure_ascii=False))

[LOG] llm_call_start: {'attempt': 1}
[LOG] llm_call_end: {'attempt': 1, 'elapsed': 2.53, 'input_tokens': 365, 'output_tokens': 36}
[LOG] parse_ok: {'category': 'laptop', 'max_price': 70000, 'sort_by': 'price_asc'}
[LOG] params_validated: {'params': {'category': 'laptop', 'max_price': 70000, 'sort_by': 'price_asc'}}
[LOG] search_done: {'found': 3}

Результат: {'category': 'laptop', 'max_price': 70000, 'sort_by': 'price_asc'} найдено: 3

--- Журнал ---
{"time": "2026-11-10T23:11:04", "step": "llm_call_start", "attempt": 1}
{"time": "2026-11-10T23:11:07", "step": "llm_call_end", "attempt": 1, "elapsed": 2.53, "input_tokens": 365, "output_tokens": 36}
{"time": "2026-11-10T23:11:07", "step": "parse_ok", "category": "laptop", "max_price": 70000, "sort_by": "price_asc"}
{"time": "2026-11-10T23:11:07", "step": "params_validated", "params": {"category": "laptop", "max_price": 70000, "sort_by": "price_asc"}}
{"time": "2026-11-10T23:11:07", "step": "search_done", "found": 3}


In [51]:
with open("results/part4_3_pipeline_log.json", "w", encoding="utf-8") as f:
    json.dump(pipeline_log, f, ensure_ascii=False, indent=2)
print(f"Сохранено {len(pipeline_log)} записей в results/part4_3_pipeline_log.json")

Сохранено 5 записей в results/part4_3_pipeline_log.json


In [50]:
def run_search_with_confirm(query: ProductQuery, auto_confirm: bool = True) -> dict:
    log("params_validated", params=query.model_dump())
    if query.category == "other":
        log("search_skipped", reason="unsupported_category")
        return {"query": query.model_dump(), "products": [], "message": "Категория не поддерживается."}
    if query.max_price is None and not auto_confirm:
        log("confirmation_required", reason="no_max_price")
        return {"query": query.model_dump(), "products": [],
                "message": "Поиск без ограничения цены. Подтвердите (y/n): "}
    products = search_products(query)
    log("search_done", found=len(products))
    return {"query": query.model_dump(), "products": products,
            "message": None if products else "Ничего не найдено."}

In [52]:
print("--- Поиск БЕЗ ограничения цены (auto_confirm=False) ---")
pipeline_log.clear()
out = run_search_with_confirm(
    ProductQuery(category="tablet", max_price=None, sort_by="price_desc"),
    auto_confirm=False,
)
print(out)
for e in pipeline_log:
    print(json.dumps(e, ensure_ascii=False))

print("\n--- Поиск БЕЗ ограничения цены (auto_confirm=True) ---")
pipeline_log.clear()
out = run_search_with_confirm(
    ProductQuery(category="tablet", max_price=None, sort_by="price_desc"),
    auto_confirm=True,
)
print(out)
for e in pipeline_log:
    print(json.dumps(e, ensure_ascii=False))

--- Поиск БЕЗ ограничения цены (auto_confirm=False) ---
[LOG] params_validated: {'params': {'category': 'tablet', 'max_price': None, 'sort_by': 'price_desc'}}
[LOG] confirmation_required: {'reason': 'no_max_price'}
{'query': {'category': 'tablet', 'max_price': None, 'sort_by': 'price_desc'}, 'products': [], 'message': 'Поиск без ограничения цены. Подтвердите (y/n): '}
{"time": "2026-11-10T23:14:30", "step": "params_validated", "params": {"category": "tablet", "max_price": null, "sort_by": "price_desc"}}
{"time": "2026-11-10T23:14:30", "step": "confirmation_required", "reason": "no_max_price"}

--- Поиск БЕЗ ограничения цены (auto_confirm=True) ---
[LOG] params_validated: {'params': {'category': 'tablet', 'max_price': None, 'sort_by': 'price_desc'}}
[LOG] search_done: {'found': 5}
{'query': {'category': 'tablet', 'max_price': None, 'sort_by': 'price_desc'}, 'products': [{'id': 17, 'name': 'Samsung Galaxy Tab S9', 'category': 'tablet', 'price': 85000}, {'id': 16, 'name': 'Apple iPad 10',

In [53]:
with open("results/part4_3_pipeline_log.json", "w", encoding="utf-8") as f:
    json.dump(pipeline_log, f, ensure_ascii=False, indent=2)
print(f"Сохранено {len(pipeline_log)} записей в results/part4_3_pipeline_log.json")

Сохранено 2 записей в results/part4_3_pipeline_log.json


In [54]:
# сначала нормальный запрос
pipeline_log.clear()
attempt_counter["n"] = 0
chain_full_logged.invoke({"query": "Найди ноутбуки дешевле 70 000 рублей"})
log_normal = list(pipeline_log)

# потом запрос без ограничения цены
pipeline_log.clear()
run_search_with_confirm(
    ProductQuery(category="tablet", max_price=None, sort_by="price_desc"),
    auto_confirm=False,
)
log_confirm = list(pipeline_log)

# сохраняем оба
with open("results/part4_3_pipeline_log.json", "w", encoding="utf-8") as f:
    json.dump({"normal_search": log_normal, "confirmation_required": log_confirm},
              f, ensure_ascii=False, indent=2)
print("Сохранено:", len(log_normal), "+", len(log_confirm), "записей")

[LOG] llm_call_start: {'attempt': 1}
[LOG] llm_call_end: {'attempt': 1, 'elapsed': 1.62, 'input_tokens': 365, 'output_tokens': 41}
[LOG] parse_ok: {'category': 'laptop', 'max_price': 70000, 'sort_by': 'price_asc'}
[LOG] params_validated: {'params': {'category': 'laptop', 'max_price': 70000, 'sort_by': 'price_asc'}}
[LOG] search_done: {'found': 3}
[LOG] params_validated: {'params': {'category': 'tablet', 'max_price': None, 'sort_by': 'price_desc'}}
[LOG] confirmation_required: {'reason': 'no_max_price'}
Сохранено: 5 + 2 записей


**воспроизводимый тест**

Нужна фейковая модель, которая возвращает заранее заданные ответы, чтобы гарантированно воспроизвести ошибку парсинга. В LangChain есть FakeListChatModel (из langchain_core.language_models.fake_chat_models)  возвращает ответы из списка по очереди.

In [55]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel

# Сценарий 1: первая попытка — битый JSON, вторая — корректный
fake_responses_ok = [
    '{"category": "laptop", "max_price": 70000,}',                 # битый
    '{"category": "laptop", "max_price": 70000, "sort_by": "price_asc"}',  # корректный
]

# Сценарий 2: все три попытки — битый JSON
fake_responses_fail = [
    '{"category": "laptop", "max_price": 70000,}',
    '{"category": "laptop", "max_price": 70000,}',
    '{"category": "laptop", "max_price": 70000,}',
]

In [56]:
def build_chain_with_fake(responses):
    fake_llm = FakeListChatModel(responses=responses)
    return prompt_structured | fake_llm | parser

# --- Без retry ---
print("=" * 60)
print("СЦЕНАРИЙ 1: битый → корректный, БЕЗ retry")
chain_no_retry = build_chain_with_fake(fake_responses_ok)
try:
    print(chain_no_retry.invoke({"query": "Найди ноутбуки дешевле 70 000"}))
except Exception as e:
    print(f"ОШИБКА: {type(e).__name__}: {str(e)[:150]}")

СЦЕНАРИЙ 1: битый → корректный, БЕЗ retry
ОШИБКА: OutputParserException: Invalid json output: {"category": "laptop", "max_price": 70000,}
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OU


In [57]:
# --- С retry ---
print("=" * 60)
print("СЦЕНАРИЙ 1: битый → корректный, С retry")
chain_with_retry = build_chain_with_fake(fake_responses_ok).with_retry(
    stop_after_attempt=3,
    retry_if_exception_type=(OutputParserException,),
)
try:
    result = chain_with_retry.invoke({"query": "Найди ноутбуки дешевле 70 000"})
    print("УСПЕХ:", result)
except Exception as e:
    print(f"ОШИБКА: {type(e).__name__}: {str(e)[:150]}")

СЦЕНАРИЙ 1: битый → корректный, С retry
УСПЕХ: category='laptop' max_price=70000 sort_by='price_asc'


In [58]:
print("=" * 60)
print("СЦЕНАРИЙ 2: три битых подряд, С retry")
chain_fail = build_chain_with_fake(fake_responses_fail).with_retry(
    stop_after_attempt=3,
    retry_if_exception_type=(OutputParserException,),
)
try:
    print(chain_fail.invoke({"query": "Найди ноутбуки дешевле 70 000"}))
except Exception as e:
    print(f"ОШИБКА после 3 попыток: {type(e).__name__}: {str(e)[:150]}")

СЦЕНАРИЙ 2: три битых подряд, С retry
ОШИБКА после 3 попыток: OutputParserException: Invalid json output: {"category": "laptop", "max_price": 70000,}
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OU


In [59]:
def safe_invoke_fake(chain, query: str) -> dict:
    try:
        return chain.invoke({"query": query})
    except OutputParserException as e:
        return {"error": f"Не удалось разобрать ответ после 3 попыток: {str(e)[:200]}"}

print(safe_invoke_fake(chain_fail, "Найди ноутбуки дешевле 70 000"))

{'error': 'Не удалось разобрать ответ после 3 попыток: Invalid json output: {"category": "laptop", "max_price": 70000,}\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE '}


In [63]:
def build_logged_chain_with_fake(responses):
    fake_llm = FakeListChatModel(responses=responses)
    return (prompt_structured | RunnableLambda(llm_step) | RunnableLambda(parse_step))

# Сценарий 1: битый → корректный
pipeline_log.clear()
attempt_counter["n"] = 0
fake_chain = build_logged_chain_with_fake(fake_responses_ok).with_retry(
    stop_after_attempt=3, retry_if_exception_type=(OutputParserException,),
) | RunnableLambda(run_search_logged)
try:
    out = fake_chain.invoke({"query": "Найди ноутбуки дешевле 70 000"})
    print("Найдено:", len(out["products"]))
except Exception as e:
    print("Ошибка:", e)

print("\n--- Журнал сценария 1 ---")
for e in pipeline_log:
    print(json.dumps(e, ensure_ascii=False))

[LOG] llm_call_start: {'attempt': 1}
[LOG] llm_call_end: {'attempt': 1, 'elapsed': 2.31, 'input_tokens': 364, 'output_tokens': 41}
[LOG] parse_ok: {'category': 'laptop', 'max_price': 70000, 'sort_by': 'price_asc'}
[LOG] params_validated: {'params': {'category': 'laptop', 'max_price': 70000, 'sort_by': 'price_asc'}}
[LOG] search_done: {'found': 3}
Найдено: 3

--- Журнал сценария 1 ---
{"time": "2026-11-10T23:25:40", "step": "llm_call_start", "attempt": 1}
{"time": "2026-11-10T23:25:42", "step": "llm_call_end", "attempt": 1, "elapsed": 2.31, "input_tokens": 364, "output_tokens": 41}
{"time": "2026-11-10T23:25:42", "step": "parse_ok", "category": "laptop", "max_price": 70000, "sort_by": "price_asc"}
{"time": "2026-11-10T23:25:42", "step": "params_validated", "params": {"category": "laptop", "max_price": 70000, "sort_by": "price_asc"}}
{"time": "2026-11-10T23:25:42", "step": "search_done", "found": 3}


In [ ]:
def make_llm_step(model):
    def step(inputs):
        attempt_counter["n"] += 1
        log("llm_call_start", attempt=attempt_counter["n"])
        resp = model.invoke(inputs)
        log("llm_call_end", attempt=attempt_counter["n"])
        return resp
    return step

def build_logged_chain_with_fake(responses):
    fake_llm = FakeListChatModel(responses=responses)
    return prompt_structured | RunnableLambda(make_llm_step(fake_llm)) | RunnableLambda(parse_step)

In [64]:
def build_logged_chain_with_fake(responses):
    fake_llm = FakeListChatModel(responses=responses)
    return (
        prompt_structured
        | RunnableLambda(make_llm_step(fake_llm))
        | RunnableLambda(parse_step)
    )

# Сценарий 1: битый → корректный, с журналом
pipeline_log.clear()
attempt_counter["n"] = 0
fake_chain = build_logged_chain_with_fake(fake_responses_ok).with_retry(
    stop_after_attempt=3, retry_if_exception_type=(OutputParserException,),
) | RunnableLambda(run_search_logged)

out = fake_chain.invoke({"query": "Найди ноутбуки дешевле 70 000"})
print("Найдено:", len(out["products"]))

print("\n--- Журнал: битый → корректный (с retry) ---")
for e in pipeline_log:
    print(json.dumps(e, ensure_ascii=False))

[LOG] llm_call_start: {'attempt': 1}
[LOG] llm_call_end: {'attempt': 1}
[LOG] parse_error: {'error_type': 'OutputParserException', 'error': 'Invalid json output: {"category": "laptop", "max_price": 70000,}\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE '}
[LOG] llm_call_start: {'attempt': 2}
[LOG] llm_call_end: {'attempt': 2}
[LOG] parse_ok: {'category': 'laptop', 'max_price': 70000, 'sort_by': 'price_asc'}
[LOG] params_validated: {'params': {'category': 'laptop', 'max_price': 70000, 'sort_by': 'price_asc'}}
[LOG] search_done: {'found': 3}
Найдено: 3

--- Журнал: битый → корректный (с retry) ---
{"time": "2026-11-10T23:26:19", "step": "llm_call_start", "attempt": 1}
{"time": "2026-11-10T23:26:19", "step": "llm_call_end", "attempt": 1}
{"time": "2026-11-10T23:26:19", "step": "parse_error", "error_type": "OutputParserException", "error": "Invalid json output: {\"category\": \"laptop\", \"max_price\": 70000,}\nFor troubleshooting

In [65]:
pipeline_log.clear()
attempt_counter["n"] = 0
fake_chain_fail = build_logged_chain_with_fake(fake_responses_fail).with_retry(
    stop_after_attempt=3, retry_if_exception_type=(OutputParserException,),
) | RunnableLambda(run_search_logged)

try:
    fake_chain_fail.invoke({"query": "Найди ноутбуки дешевле 70 000"})
except OutputParserException as e:
    print(f"Ошибка после 3 попыток: {str(e)[:120]}")

print("\n--- Журнал: три битых подряд ---")
for e in pipeline_log:
    print(json.dumps(e, ensure_ascii=False))

[LOG] llm_call_start: {'attempt': 1}
[LOG] llm_call_end: {'attempt': 1}
[LOG] parse_error: {'error_type': 'OutputParserException', 'error': 'Invalid json output: {"category": "laptop", "max_price": 70000,}\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE '}
[LOG] llm_call_start: {'attempt': 2}
[LOG] llm_call_end: {'attempt': 2}
[LOG] parse_error: {'error_type': 'OutputParserException', 'error': 'Invalid json output: {"category": "laptop", "max_price": 70000,}\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE '}
[LOG] llm_call_start: {'attempt': 3}
[LOG] llm_call_end: {'attempt': 3}
[LOG] parse_error: {'error_type': 'OutputParserException', 'error': 'Invalid json output: {"category": "laptop", "max_price": 70000,}\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE '}
Ошибка после 3 попыток: Invalid json output: {"categ

In [66]:
pipeline_log.clear(); attempt_counter["n"] = 0
build_logged_chain_with_fake(fake_responses_ok).with_retry(
    stop_after_attempt=3, retry_if_exception_type=(OutputParserException,),
) | RunnableLambda(run_search_logged)

RunnableRetry(bound=ChatPromptTemplate(input_variables=['query'], input_types={}, partial_variables={'format_instructions': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"category": {"description": "Категория товара. Если категория не входит в список поддерживаемых, используй other.", "enum": ["laptop", "smartphone", "tablet", "other"], "title": "Category", "type": "string"}, "max_price": {"anyOf": [{"minimum": 0, "type": "integer"}, {"type": "null"}], "default": null, "description": "Верхняя граница цены в рублях. Если ограничения нет — null.", "title": "Max Price

In [67]:
import json

def run_fake_scenario(responses, label):
    pipeline_log.clear()
    attempt_counter["n"] = 0
    chain = build_logged_chain_with_fake(responses).with_retry(
        stop_after_attempt=3, retry_if_exception_type=(OutputParserException,),
    ) | RunnableLambda(run_search_logged)
    error = None
    out = None
    try:
        out = chain.invoke({"query": "Найди ноутбуки дешевле 70 000"})
    except OutputParserException as e:
        error = str(e)[:300]
    return {"label": label, "error": error, "found": len(out["products"]) if out else None,
            "log": list(pipeline_log)}

results_44 = [
    run_fake_scenario(fake_responses_ok, "битый → корректный (retry)"),
    run_fake_scenario(fake_responses_fail, "три битых подряд (retry)"),
]

with open("results/part4_4_fake_model_tests.json", "w", encoding="utf-8") as f:
    json.dump(results_44, f, ensure_ascii=False, indent=2)
print("Сохранено в results/part4_4_fake_model_tests.json")

[LOG] llm_call_start: {'attempt': 1}
[LOG] llm_call_end: {'attempt': 1}
[LOG] parse_error: {'error_type': 'OutputParserException', 'error': 'Invalid json output: {"category": "laptop", "max_price": 70000,}\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE '}
[LOG] llm_call_start: {'attempt': 2}
[LOG] llm_call_end: {'attempt': 2}
[LOG] parse_ok: {'category': 'laptop', 'max_price': 70000, 'sort_by': 'price_asc'}
[LOG] params_validated: {'params': {'category': 'laptop', 'max_price': 70000, 'sort_by': 'price_asc'}}
[LOG] search_done: {'found': 3}
[LOG] llm_call_start: {'attempt': 1}
[LOG] llm_call_end: {'attempt': 1}
[LOG] parse_error: {'error_type': 'OutputParserException', 'error': 'Invalid json output: {"category": "laptop", "max_price": 70000,}\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE '}
[LOG] llm_call_start: {'attempt': 2}
[LOG] llm_call_end: {'attempt': 2}
[LOG] 

## Часть 5. Уточняющие запросы

In [68]:
# Текущее состояние — последний валидный ProductQuery
current_query = None

def update_query(new_message: str, prev: ProductQuery | None) -> ProductQuery:
    """Обновляет параметры поиска с учётом предыдущего состояния."""
    prev_json = prev.model_dump_json() if prev else "нет (первый запрос)"
    prompt_update = ChatPromptTemplate.from_messages([
        ("system",
         "Ты — ассистент интернет-магазина. У тебя есть текущие параметры поиска "
         "(JSON) и новая реплика пользователя. Обнови параметры с учётом реплики. "
         "Если пользователь не упоминает поле — сохрани его прежнее значение.\n\n"
         "Текущие параметры: {prev_json}\n\n{format_instructions}"),
        ("human", "{query}"),
    ]).partial(format_instructions=format_instructions)

    chain_update = prompt_update | llm | parser
    return chain_update.invoke({"prev_json": prev_json, "query": new_message})

In [70]:
dialog = [
    "Покажи ноутбуки до 80 тысяч",
    "А сначала дешёвые?",
    "А что есть среди смартфонов?",
]

current_query = None
for msg in dialog:
    print(f"\n{'='*60}\nПользователь: {msg}")
    print(f"Предыдущее состояние: {current_query.model_dump() if current_query else None}")
    current_query = update_query(msg, current_query)
    print(f"Новое состояние:      {current_query.model_dump()}")
    result = run_search(current_query)
    print(f"Найдено: {len(result['products'])}")
    for p in result["products"]:
        print(f"  {p['name']} — {p['price']} ₽")


Пользователь: Покажи ноутбуки до 80 тысяч
Предыдущее состояние: None
Новое состояние:      {'category': 'laptop', 'max_price': 80000, 'sort_by': 'price_desc'}
Найдено: 4
  HP Pavilion 14 — 75000 ₽
  Acer Aspire 5 — 62000 ₽
  ASUS VivoBook 15 — 58000 ₽
  Lenovo IdeaPad 3 — 45000 ₽

Пользователь: А сначала дешёвые?
Предыдущее состояние: {'category': 'laptop', 'max_price': 80000, 'sort_by': 'price_desc'}
Новое состояние:      {'category': 'laptop', 'max_price': 80000, 'sort_by': 'price_asc'}
Найдено: 4
  Lenovo IdeaPad 3 — 45000 ₽
  ASUS VivoBook 15 — 58000 ₽
  Acer Aspire 5 — 62000 ₽
  HP Pavilion 14 — 75000 ₽

Пользователь: А что есть среди смартфонов?
Предыдущее состояние: {'category': 'laptop', 'max_price': 80000, 'sort_by': 'price_asc'}
Новое состояние:      {'category': 'smartphone', 'max_price': 80000, 'sort_by': 'price_asc'}
Найдено: 4
  Xiaomi Redmi Note 12 — 18000 ₽
  Samsung Galaxy A54 — 32000 ₽
  Google Pixel 7a — 45000 ₽
  Apple iPhone 14 — 78000 ₽


In [ ]:
import json
dialog_records = []
current_query = None
for msg in dialog:
    prev = current_query.model_dump() if current_query else None
    current_query = update_query(msg, current_query)
    result = run_search(current_query)
    dialog_records.append({
        "message": msg,
        "prev_state": prev,
        "new_state": current_query.model_dump(),
        "found": len(result["products"]),
    })
with open("results/part5_1_state_dialog.json", "w", encoding="utf-8") as f:
    json.dump(dialog_records, f, ensure_ascii=False, indent=2)
print("Сохранено в results/part5_1_state_dialog.json")